# GIS First Map - Points, Interpolation and Choropleth (No Install)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/civil-engineering-teaching-portfolio/blob/main/code/notebooks/gis_intro_demo.ipynb)

**Course:** Introductory GIS · **Author:** Syed Rafi Ahmed · **License:** MIT

A first-week teaching demo: render survey points, build an interpolated
surface (Inverse Distance Weighting), draw a buffer and a synthetic
choropleth - using only NumPy, pandas and Matplotlib, so it runs anywhere,
including Colab, with zero installation.

Download the CSV template in the Data cell and use its Upload button to map your own survey points.

In [ ]:
# ---- Setup ----
%pip install -q numpy pandas matplotlib


## Data - use the demo, or upload your own

The notebook **runs automatically**: *Runtime > Run all* completes end-to-end on the
built-in synthetic survey points, with no interaction needed.

**To map your own survey instead** (three steps):

1. **Get the template** - the next cell creates `upload_template_points.csv`, shows a
   preview of it, and in Colab downloads it to your computer automatically.
2. **Fill in your points** - keep the header row exactly as it is, delete the example
   rows, and add yours (one row per survey point).
3. **Upload** - click the **Upload** button at the bottom of the next cell and pick your
   CSV, then re-run the following cell: your points replace the demo points and a preview
   is shown so you can confirm what was loaded. If you upload nothing, the demo points are kept.

Required columns (spelling matters; extra columns are ignored): `point_id, x_m, y_m, elevation_m`


In [ ]:
# ---- Data: template download, upload button, and automatic demo fallback ----
import io

import numpy as np
import pandas as pd

POINT_COLS = {"x_m", "y_m", "elevation_m"}

DEMO_POINTS_CSV = """point_id,x_m,y_m,elevation_m
1,40,80,42.6
2,150,60,45.1
3,260,110,47.8
4,380,70,52.3
5,500,120,55.9
6,620,80,61.4
7,740,130,58.7
8,860,90,52.2
9,950,140,49.8
10,90,220,44.9
11,210,260,49.6
12,330,230,54.1
13,450,280,58.3
14,570,240,63.7
15,690,290,60.2
16,810,250,54.6
17,930,300,51.1
18,60,380,47.3
19,180,420,52.8
20,300,390,57.4
21,420,440,62.9
22,540,400,66.8
23,660,450,62.1
24,780,410,56.3
25,900,460,53.5
26,110,560,50.2
27,240,600,55.7
28,360,570,60.4
29,480,620,64.6
30,600,580,59.9
31,720,630,55.4
32,840,590,52.7
33,960,640,50.9
34,30,660,46.4
35,130,300,50.8
36,880,350,57.9
37,520,520,63.1
38,280,520,58.6
39,760,560,58.2
40,40,480,51.5
"""

TEMPLATE_POINTS_CSV = '''point_id,x_m,y_m,elevation_m
101,120,150,48.0
102,340,260,55.0
103,700,410,58.0
'''

# -- 1) Build the downloadable template and preview what to upload --
template_points = pd.read_csv(io.StringIO(TEMPLATE_POINTS_CSV))
template_points.to_csv("upload_template_points.csv", index=False)
print("PREVIEW - upload_template_points.csv (one row per survey point):")
print(template_points.to_string(index=False))
print()

# -- 2) In Colab, download the template automatically --
try:
    from google.colab import files  # type: ignore
    files.download("upload_template_points.csv")
    print("Template downloaded to your computer (and saved next to the notebook).")
except Exception:
    print("Template saved next to the notebook: upload_template_points.csv")

# -- 3) Show the upload button (non-blocking: Run all keeps going) --
uploader = None
try:
    import ipywidgets as widgets  # type: ignore
    uploader = widgets.FileUpload(accept=".csv", multiple=True)
    print("\nUpload your filled-in CSV here (or leave empty to keep the demo points):")
    try:
        from IPython.display import display as _display
        _display(uploader)
    except Exception:
        pass
except Exception as exc:
    print(f"\nUpload button unavailable here ({exc}). In Colab you can instead run in a new cell:")
    print("    from google.colab import files; files.upload()")


In [ ]:
# ---- Read the upload (if any), otherwise keep the demo points ----
# After choosing a file with the Upload button above, re-run this cell
# (or Runtime > Run all) to load it.

def _widget_files(value):
    """Normalise ipywidgets 7 (dict) and 8 (tuple of dicts) upload values."""
    if not value:
        return []
    if isinstance(value, dict):
        return [(str(name), bytes(mem)) for name, mem in value.items()]
    return [(str(item.get("name", f"upload_{i}.csv")), bytes(item.get("content", b"")))
            for i, item in enumerate(value)]


uploaded_pts = None
uploads = _widget_files(uploader.value) if uploader is not None else []
for name, raw in uploads:
    try:
        frame = pd.read_csv(io.BytesIO(raw))
    except Exception as exc:
        print(f"  skipped {name}: not readable as CSV ({exc})")
        continue
    if POINT_COLS <= set(frame.columns):
        uploaded_pts = frame
        print(f"  point data loaded from: {name}")
    else:
        print(f"  skipped {name}: missing x_m / y_m / elevation_m columns")

if uploaded_pts is None and uploads:
    print("  no usable file found - keeping the demo points.")
if not uploads:
    print("No file uploaded - running on the built-in demo points.")

if uploaded_pts is not None:
    pts = uploaded_pts.copy()
    print(f"\nYOUR point data: {len(pts)} rows - preview:")
    print(pts.head(6).to_string(index=False))
else:
    pts = pd.read_csv(io.StringIO(DEMO_POINTS_CSV))

pts.head()


In [ ]:
# ---- Inverse Distance Weighting interpolation ----
def idw_interpolate(x, y, v, gx, gy, power=2.0):
    d = np.sqrt((gx[None, :, :] - np.asarray(x)[:, None, None]) ** 2
                + (gy[None, :, :] - np.asarray(y)[:, None, None]) ** 2)
    d[d < 1e-9] = 1e-9
    w = 1.0 / d ** power
    return (w * np.asarray(v)[:, None, None]).sum(axis=0) / w.sum(axis=0)


gx, gy = np.meshgrid(np.linspace(0, 1000, 160), np.linspace(0, 700, 120))
z = idw_interpolate(pts["x_m"], pts["y_m"], pts["elevation_m"], gx, gy)
print(f"Surface range: {z.min():.1f} - {z.max():.1f} m")


In [ ]:
# ---- Map: shaded surface, contours, points and a 120 m buffer ----
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
ls = LightSource(azdeg=315, altdeg=45)
rgb = ls.shade(z, cmap=plt.get_cmap("YlGnBu"), vert_exag=0.1, blend_mode="soft")
ax.imshow(rgb, origin="lower", extent=(0, 1000, 0, 700), aspect="auto")
cs = ax.contour(gx, gy, z, levels=8, colors="white", linewidths=0.5, alpha=0.5)
ax.clabel(cs, fmt="%.0f m", fontsize=7)
ax.scatter(pts["x_m"], pts["y_m"], c="white", edgecolors="black", s=28,
           label="survey points", zorder=3)
hi = pts.loc[pts["elevation_m"].idxmax()]
ax.add_patch(plt.Circle((hi["x_m"], hi["y_m"]), 120, fill=False,
                        color="#c25b34", linewidth=2, label="120 m buffer"))
ax.set_xlim(0, 1000); ax.set_ylim(0, 700)
ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
ax.set_title("IDW surface, contours and a buffer around the highest point")
ax.legend(frameon=False, loc="upper left")
plt.show()


In [ ]:
# ---- Synthetic choropleth: attribute mapped to fill colour ----
rng = np.random.default_rng(7)
scores = rng.integers(1, 5, size=(4, 6)).astype(float)
cmap = plt.get_cmap("YlGnBu")
norm = plt.Normalize(vmin=1, vmax=4)
fig, ax = plt.subplots(figsize=(7.2, 4.2), constrained_layout=True)
for i in range(4):
    for j in range(6):
        v = scores[i, j]
        ax.add_patch(plt.Rectangle((j * 100, 560 - i * 140), 100, 140,
                                   facecolor=cmap(norm(v)), edgecolor="white"))
        ax.text(j * 100 + 50, 560 - i * 140 + 70, f"{v:.0f}",
                ha="center", va="center", color="#22303a")
ax.set_xlim(0, 600); ax.set_ylim(0, 560)
ax.set_title("Synthetic zone choropleth (land-use score)")
ax.set_xticks([]); ax.set_yticks([])
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, shrink=0.85, ticks=[1, 2, 3, 4], label="score")
plt.show()


## Self-check quiz

<details><summary><b>Q1.</b> Why does IDW reproduce the sample points exactly?</summary>
At a sample location the distance term of that point dominates (d -> 0 makes its
weight explode), so the interpolated value equals the measured value.
</details>

<details><summary><b>Q2.</b> What happens with power = 1 vs power = 4?</summary>
Lower power smooths more (distant points matter), higher power localises - the
surface becomes spikier around samples. Try it: change the power argument.
</details>

<details><summary><b>Q3.</b> Is the buffer drawn here a true geodesic buffer?</summary>
No - it is a planar circle on a local metric grid, which is exactly why real GIS
software and a stated CRS are needed for defensible buffers.
</details>


## Limitations

- This demo teaches *concepts* (layers, interpolation, buffers, choropleth).
  Real workflows use QGIS / ArcGIS / geopandas with proper CRS handling.
- Default points are synthetic; upload your own CSV (columns `point_id, x_m,
  y_m, elevation_m`) to map real local data.

Portfolio author: Syed Rafi Ahmed - civil-engineering-teaching-portfolio (MIT).
